In [1]:
import pandas as pd

# Load splits
train_df = pd.read_csv("train.csv")
val_df = pd.read_csv("validation.csv")
test_df = pd.read_csv("test.csv")

print("Train shape:", train_df.shape)
print("Validation shape:", val_df.shape)
print("Test shape:", test_df.shape)

train_df.head()

Train shape: (3900, 2)
Validation shape: (836, 2)
Test shape: (836, 2)


,label,message
0,0,2 and half years i missed your friendship:-)
1,0,Can you say what happen
2,0,Went to ganesh dress shop
3,0,Heart is empty without love.. Mind is empty wi...
4,1,Not heard from U4 a while. Call 4 rude chat pr...


In [2]:
from sklearn.feature_extraction.text import TfidfVectorizer

X_train_text = train_df["message"]
y_train = train_df["label"]

X_val_text = val_df["message"]
y_val = val_df["label"]

vectorizer = TfidfVectorizer(stop_words="english", max_features=5000)

X_train = vectorizer.fit_transform(X_train_text)
X_val = vectorizer.transform(X_val_text)

print("Feature shape:", X_train.shape)

Feature shape: (3900, 5000)


In [3]:
import mlflow

# Force MLflow to use local folder tracking
mlflow.set_tracking_uri("file:./mlruns")

mlflow.set_experiment("SMS_Spam_Models")

C:\Users\user\AppData\Roaming\Python\Python314\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\user\AppData\Roaming\Python\Python314\site-packages\mlflow\tracking\_tracking_service\utils.py:178: FutureWarning: The filesystem tracking backend (e.g., './mlruns') will be deprecated in February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://github.com/mlflow/mlflow/issues/18534 for more details and migration guidance. For migrating existing data, https://github.com/mlflow/mlflow-export-import can be used.
  return FileStore(store_uri, store_uri)


<Experiment: artifact_location='file:c:/Users/user/Desktop/Applied ML/Assignment 2/mlruns/919734695492677161', creation_time=1771177723191, experiment_id='919734695492677161', last_update_time=1771177723191, lifecycle_stage='active', name='SMS_Spam_Models', tags={}>

In [4]:
# Set experiment
mlflow.set_experiment("SMS_Spam_Models")

<Experiment: artifact_location='file:c:/Users/user/Desktop/Applied ML/Assignment 2/mlruns/919734695492677161', creation_time=1771177723191, experiment_id='919734695492677161', last_update_time=1771177723191, lifecycle_stage='active', name='SMS_Spam_Models', tags={}>

In [5]:
import mlflow
import mlflow.sklearn

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score

### Model 1 - Logistic Regression

In [6]:
# Model 1: Logistic Regression

with mlflow.start_run(run_name="Logistic_Regression"):
    
    # Model
    model = LogisticRegression(max_iter=1000)
    model.fit(X_train, y_train)
    
    # Validation predictions (probabilities)
    y_val_probs = model.predict_proba(X_val)[:, 1]
    
    # AUCPR
    aucpr = average_precision_score(y_val, y_val_probs)
    
    # Log parameters
    mlflow.log_param("model", "LogisticRegression")
    mlflow.log_param("max_iter", 1000)
    
    # Log metric
    mlflow.log_metric("AUCPR", aucpr)
    
    # Log model artifact
    mlflow.sklearn.log_model(
        model,
        artifact_path="model",
        registered_model_name="SMS_SPAM_Logistic"
    )
    
    print("Logistic Regression AUCPR:", aucpr)


2026/02/15 23:22:10 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
C:\Users\user\AppData\Roaming\Python\Python314\site-packages\mlflow\models\model.py:1209: FutureWarning: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization.The recommended safe alternative is the 'skops' format.
  flavor.save_model(path=local_path, mlflow_model=mlflow_model, **kwargs)


Logistic Regression AUCPR: 0.9405178462298899


C:\Users\user\AppData\Roaming\Python\Python314\site-packages\mlflow\tracking\_model_registry\utils.py:216: FutureWarning: The filesystem model registry backend (e.g., './mlruns') will be deprecated in February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://github.com/mlflow/mlflow/issues/18534 for more details and migration guidance. For migrating existing data, https://github.com/mlflow/mlflow-export-import can be used.
  return FileStore(store_uri)
Successfully registered model 'SMS_SPAM_Logistic'.
Created version '1' of model 'SMS_SPAM_Logistic'.


## Model 2 :- Naive Bayes

In [7]:
from sklearn.naive_bayes import MultinomialNB

# Model 2: Naive Bayes

with mlflow.start_run(run_name="Naive_Bayes"):
    
    model = MultinomialNB()
    model.fit(X_train, y_train)
    
    # Probabilities
    y_val_probs = model.predict_proba(X_val)[:, 1]
    
    # AUCPR
    aucpr = average_precision_score(y_val, y_val_probs)
    
    # Log parameters
    mlflow.log_param("model", "MultinomialNB")
    
    # Log metric
    mlflow.log_metric("AUCPR", aucpr)
    
    # Log and register model
    mlflow.sklearn.log_model(
        model,
        artifact_path="model",
        registered_model_name="SMS_SPAM_NaiveBayes"
    )
    
    print("Naive Bayes AUCPR:", aucpr)


2026/02/15 23:23:53 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
C:\Users\user\AppData\Roaming\Python\Python314\site-packages\mlflow\models\model.py:1209: FutureWarning: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization.The recommended safe alternative is the 'skops' format.
  flavor.save_model(path=local_path, mlflow_model=mlflow_model, **kwargs)


Naive Bayes AUCPR: 0.9655311133754256


Successfully registered model 'SMS_SPAM_NaiveBayes'.
Created version '1' of model 'SMS_SPAM_NaiveBayes'.


## Model 3:- Random Forest Classifier

In [8]:
from sklearn.ensemble import RandomForestClassifier

# Model 3: Random Forest

with mlflow.start_run(run_name="Random_Forest"):
    
    model = RandomForestClassifier(
        n_estimators=100,
        random_state=42,
        n_jobs=-1
    )
    
    model.fit(X_train, y_train)
    
    # Probabilities
    y_val_probs = model.predict_proba(X_val)[:, 1]
    
    # AUCPR
    aucpr = average_precision_score(y_val, y_val_probs)
    
    # Log parameters
    mlflow.log_param("model", "RandomForest")
    mlflow.log_param("n_estimators", 100)
    
    # Log metric
    mlflow.log_metric("AUCPR", aucpr)
    
    # Log and register model
    mlflow.sklearn.log_model(
        model,
        artifact_path="model",
        registered_model_name="SMS_SPAM_RandomForest"
    )
    
    print("Random Forest AUCPR:", aucpr)


2026/02/15 23:25:17 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
C:\Users\user\AppData\Roaming\Python\Python314\site-packages\mlflow\models\model.py:1209: FutureWarning: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization.The recommended safe alternative is the 'skops' format.
  flavor.save_model(path=local_path, mlflow_model=mlflow_model, **kwargs)


Random Forest AUCPR: 0.9670617629179232


Successfully registered model 'SMS_SPAM_RandomForest'.
Created version '1' of model 'SMS_SPAM_RandomForest'.


In [9]:
import mlflow

# Get experiment
experiment = mlflow.get_experiment_by_name("SMS_Spam_Models")
experiment_id = experiment.experiment_id

# Search all runs
runs = mlflow.search_runs(experiment_ids=[experiment_id])

# Select relevant columns
results = runs[["tags.mlflow.runName", "metrics.AUCPR"]]

# Rename for clarity
results.columns = ["Model", "AUCPR"]

print("Model Performance (Validation AUCPR)")
print("="*40)
print(results.to_string(index=False))


Model Performance (Validation AUCPR)
              Model    AUCPR
      Random_Forest 0.967062
        Naive_Bayes 0.965531
Logistic_Regression 0.940518
